# Coronary Artery Disease Survival Prediction

Exploratory data analysis, following the notebook-based workflow of the HDB price predictor. Run from the repository root with the Python 3.11 environment. The source CSV remains local; only aggregate outputs are shown.

Assumption: `Survive=1/Yes` means survival. Training uses non-survival as the positive class. There is no documented follow-up horizon.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from main import load_config
from src.data_preparation import DataPreparation
from sklearn.model_selection import StratifiedGroupKFold

config = load_config()
preparation = DataPreparation(config)
raw, risk, groups, audit = preparation.load_data(config["file_path"])
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}; unique IDs: {raw.ID.nunique():,}")

Rows: 15,000; columns: 16; unique IDs: 14,042


## 1. Dataset structure and quality

Schema checks use the full file. Outcome relationships and plots below use only the training partition.

In [2]:
display(pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(), "distinct": raw.nunique()}))
display(pd.Series({k: audit[k] for k in ["exact_duplicate_rows", "duplicate_cleaned_predictors", "negative_age_count", "ids_with_conflicting_outcomes"]}, name="count"))

,dtype,missing,distinct
ID,object,0,14042
Survive,object,0,4
Gender,object,0,2
Smoke,object,0,4
Diabetes,object,0,3
Age,int64,0,87
Ejection Fraction,object,0,5
Sodium,int64,0,27
Creatinine,float64,499,40
Platelets,float64,0,176


exact_duplicate_rows               0
duplicate_cleaned_predictors       2
negative_age_count               430
ids_with_conflicting_outcomes    375
Name: count, dtype: int64

## 2. Patient-aware training partition

Repeated IDs and exact copies of cleaned predictors stay together. The held-out rows are not used for exploratory outcome analysis.

In [3]:
splitter = StratifiedGroupKFold(n_splits=config["holdout_folds"], shuffle=True, random_state=config["random_state"])
train_idx, test_idx = next(splitter.split(raw, risk, groups))
assert not set(groups[train_idx]) & set(groups[test_idx])
train = raw.iloc[train_idx].copy()
clean = preparation.clean_data(train)
train_risk = risk[train_idx]
print(f"Training rows: {len(train_idx):,}; held-out rows: {len(test_idx):,}")

Training rows: 12,003; held-out rows: 2,997


## 3. Outcome balance

In [4]:
counts = pd.Series(train_risk).value_counts().sort_index().rename(index={0: "Survival", 1: "Non-survival"})
counts.plot.bar(rot=0, color=["#2a9d8f", "#e76f51"], title="Training outcome counts")
plt.ylabel("Records")
plt.tight_layout()
plt.show()
display(counts.to_frame("count"))

C:\Users\jheng\AppData\Local\Temp\ipykernel_22636\1381344847.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,count
Survival,3852
Non-survival,8151


## 4. Cleaning and missing values

Negative ages are treated as unknown rather than corrected by guessing. Category case, whitespace and abbreviations are normalized. Imputation is learned later, separately within each training fold.

In [5]:
display(clean.isna().sum().to_frame("missing_after_cleaning"))
for column in config["nominal_features"]:
    print(column, sorted(clean[column].dropna().unique()))

,missing_after_cleaning
Age,354
Sodium,0
Creatinine,395
Platelets,0
Creatine phosphokinase,0
Blood Pressure,0
Hemoglobin,0
Height,0
Weight,0
Gender,0


Gender ['female', 'male']
Smoke ['no', 'yes']
Diabetes ['diabetes', 'normal', 'pre-diabetes']
Ejection Fraction ['high', 'low', 'normal']


## 5. Numerical feature distributions

In [6]:
display(clean[config["numerical_features"]].describe().T)
clean[config["numerical_features"]].hist(figsize=(13, 10), bins=25, color="#457b9d")
plt.suptitle("Training feature distributions")
plt.tight_layout()
plt.show()

,count,mean,std,min,25%,50%,75%,max
Age,11649.0,60.897244,11.924132,40.0,51.0,60.0,70.0,95.0
Sodium,12003.0,136.623344,4.373632,113.0,134.0,137.0,140.0,148.0
Creatinine,11608.0,1.414879,1.058797,0.5,0.9,1.1,1.4,9.4
Platelets,12003.0,262596.770492,97652.874019,25100.0,211000.0,262000.0,304000.0,850000.0
Creatine phosphokinase,12003.0,584.068233,963.100992,23.0,115.0,250.0,582.0,7861.0
Blood Pressure,12003.0,103.498042,39.711334,40.0,70.0,100.0,137.0,179.0
Hemoglobin,12003.0,12.913263,2.496826,9.0,10.7,12.7,15.1,17.5
Height,12003.0,159.469383,17.365470,130.0,144.0,160.0,174.0,189.0
Weight,12003.0,69.411980,25.357756,19.0,50.0,67.0,87.0,141.0


C:\Users\jheng\AppData\Local\Temp\ipykernel_22636\1505400756.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Feature relationships with the recorded outcome

These aggregate associations do not show that a feature causes survival or that a treatment will help.

In [7]:
exploration = clean.assign(non_survival=train_risk)
display(exploration.groupby("non_survival")[config["numerical_features"]].mean().T)
correlation = exploration[config["numerical_features"] + ["non_survival"]].corr()
fig, ax = plt.subplots(figsize=(10, 8))
plot = ax.imshow(correlation, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(correlation)), correlation.columns, rotation=90)
ax.set_yticks(range(len(correlation)), correlation.index)
fig.colorbar(plot, ax=ax, label="Pearson correlation")
ax.set_title("Training-only numeric correlations")
plt.tight_layout()
plt.show()

non_survival,0,1
Age,65.309803,58.799113
Sodium,135.251817,137.271500
Creatinine,1.876015,1.185857
Platelets,254907.507323,266230.562876
Creatine phosphokinase,645.700415,554.942093
Blood Pressure,107.092420,101.799411
Hemoglobin,12.697508,13.015225
Height,159.723780,159.349160
Weight,84.793614,62.142927


C:\Users\jheng\AppData\Local\Temp\ipykernel_22636\1250592148.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Reproducible model comparison

Run `python main.py` to train and evaluate all models. Configuration is in `src/config.yaml`; preprocessing is in `src/data_preparation.py`; model fitting and evaluation are in `src/model_training.py`. The next cell reads the saved aggregate results without retraining or choosing models from test scores.

In [8]:
report_dir = Path(config["output_dir"])
if (report_dir / "results.json").exists():
    display(pd.read_csv(report_dir / "cross_validation.csv"))
    display(pd.read_csv(report_dir / "test_metrics.csv"))
    results = json.loads((report_dir / "results.json").read_text())
    print("Selected model:", results["selected_model"])
    print("Training-selected risk threshold:", round(results["risk_threshold"], 2))
else:
    print("Run python main.py first to generate the comparison.")

,model,cv_auc,cv_auc_sd,cv_ap,cv_ap_sd,cv_brier
0,Random forest,0.999998,0.000003,0.999999,0.000001,0.007166
1,Gradient boosting,0.999991,0.000011,0.999996,0.000005,0.006549
2,Logistic regression,0.877725,0.004824,0.934272,0.003180,0.126412
3,Dummy baseline,0.500000,0.000000,0.679086,0.008984,0.217965


,model,roc_auc,average_precision,accuracy,balanced_accuracy,risk_recall,risk_precision,risk_f2,survival_recall,brier,tn,fp,fn,tp
0,Dummy baseline,0.500000,0.678679,0.678679,0.500000,1.000000,0.678679,0.913500,0.000000,0.218074,0,963,0,2034
1,Logistic regression,0.872426,0.932797,0.825492,0.780944,0.905605,0.847676,0.893394,0.656282,0.130465,632,331,192,1842
2,Random forest,0.999999,1.000000,0.999666,0.999481,1.000000,0.999509,0.999902,0.998962,0.005547,962,1,0,2034
3,Gradient boosting,0.999996,0.999998,0.999333,0.998962,1.000000,0.999018,0.999803,0.997923,0.006111,961,2,0,2034


Selected model: Random forest
Training-selected risk threshold: 0.55


## 8. Interpretation and limitations

Random forest is the nominal winner in the committed default experiment, with gradient boosting practically tied. Near-perfect scores warrant checking dataset construction, outcome-derived predictors and repeated underlying source records. Conflicting labels for repeated IDs, unknown units/provenance and an unspecified follow-up horizon limit interpretation. See `reports/REPORT.md` for the complete evaluation. This notebook does not prescribe medical treatment.